# ConnectaTel: consumo, calidad de datos y segmentación
**Mario Alberto Vivero · Proyecto académico**

Versión revisada del código. No ejecutada con las fuentes: solo se recibió el notebook, no los tres CSV. Coloca `plans.csv`, `users_latam.csv` y `usage.csv` en `data/`. Las cifras históricas del README no son salidas de esta versión.

Objetivo: construir perfiles de uso comparables y detectar registros que requieren revisión antes de proponer cambios de plan. El corte analítico del ejercicio es el 31 de diciembre de 2024.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd()
if not (ROOT/'data').is_dir(): ROOT = ROOT.parent
files = ['plans.csv','users_latam.csv','usage.csv']
missing = [name for name in files if not (ROOT/'data'/name).is_file()]
if missing:
    raise FileNotFoundError('Fuentes requeridas: ' + ', '.join(missing))
plans = pd.read_csv(ROOT/'data/plans.csv')
users = pd.read_csv(ROOT/'data/users_latam.csv')
usage = pd.read_csv(ROOT/'data/usage.csv')
for name, frame in [('plans',plans),('users',users),('usage',usage)]:
    print(name, frame.shape, '\n', frame.isna().sum())


## Claves, fechas y calidad
Se detiene el análisis ante claves duplicadas o referencias desconocidas. No se eliminan duplicados arbitrariamente. Las fechas de registro posteriores al corte se marcan como desconocidas y se conserva una bandera. La ausencia de fecha de baja se conserva sin imputar; su significado depende de la documentación del negocio.


In [ ]:
for frame,key in [(users,'user_id'),(usage,'id'),(plans,'plan_name')]:
    if frame[key].isna().any() or frame[key].duplicated().any():
        raise ValueError(f'Clave nula o duplicada: {key}')
if not usage['user_id'].isin(users['user_id']).all():
    raise ValueError('Existen eventos de usuarios sin correspondencia.')
if not usage['type'].isin(['call','text']).all():
    raise ValueError('Tipos de evento desconocidos.')
if not users['plan'].isin(plans['plan_name']).all():
    raise ValueError('Hay planes no encontrados en catálogo.')
for frame,col in [(users,'reg_date'),(users,'churn_date'),(usage,'date')]:
    original = frame[col].copy()
    frame[col] = pd.to_datetime(original, format='mixed', errors='coerce')
    print(col, 'Fechas no interpretables:', int((original.notna() & frame[col].isna()).sum()))
users['registro_fuera_corte'] = users['reg_date'].dt.year.gt(2024)
print('Registros fuera de corte:', users['registro_fuera_corte'].sum())
users.loc[users['registro_fuera_corte'],'reg_date'] = pd.NaT
users['city'] = users['city'].astype('string').str.strip().replace({'?':pd.NA,'':pd.NA})
users['ciudad_ausente'] = users['city'].isna()
users['city'] = users['city'].fillna('Sin dato')
users['age'] = pd.to_numeric(users['age'], errors='coerce')
users['edad_imputada'] = users['age'].eq(-999) | users['age'].isna()
users.loc[users['age'].eq(-999),'age'] = np.nan
if users['age'].dropna().le(0).any():
    raise ValueError('Revisar edades no positivas adicionales.')
median_age = users['age'].median()
if pd.isna(median_age): raise ValueError('No hay edades válidas para calcular mediana.')
users['age'] = users['age'].fillna(median_age)
print('Edades imputadas:', users['edad_imputada'].sum())
print('Ciudades ausentes o sentinel:', users['ciudad_ausente'].sum())


## Eventos válidos y agregación
Los nulos estructurales se revisan por tipo; su patrón no prueba formalmente un mecanismo MAR. Duración en mensajes y longitud en llamadas son campos no aplicables que deben investigarse si contienen valores. Los minutos se suman exclusivamente sobre llamadas.

Los eventos sin fecha o fuera de 2024 se excluyen de esta agregación temporal y se contabilizan. Esto cambia el alcance frente al original, que agregaba todas las filas. Sin eventos válidos se etiqueta «Sin actividad observada», no «Alto uso» ni una afirmación de consumo real cero.


In [ ]:
for col in ['duration','length']:
    usage[col] = pd.to_numeric(usage[col], errors='raise')
    if usage[col].dropna().lt(0).any(): raise ValueError(f'Valores negativos en {col}')
print(usage.groupby('type')[['duration','length']].agg(lambda s:s.isna().mean()))
print('Textos con duración:', int((usage['type'].eq('text') & usage['duration'].notna()).sum()))
print('Llamadas con longitud:', int((usage['type'].eq('call') & usage['length'].notna()).sum()))
valid_date = usage['date'].dt.year.eq(2024)
print('Eventos excluidos por fecha:', int((~valid_date).sum()))
u = usage.loc[valid_date].copy()
u['is_call'] = u['type'].eq('call').astype(int)
u['is_text'] = u['type'].eq('text').astype(int)
u['minutos_solo_llamadas'] = u['duration'].where(u['type'].eq('call'),0)
u['duracion_faltante'] = u['type'].eq('call') & u['duration'].isna()
agg = u.groupby('user_id').agg(
    cant_mensajes=('is_text','sum'),cant_llamadas=('is_call','sum'),
    cant_minutos_llamada=('minutos_solo_llamadas','sum'),
    llamadas_sin_duracion=('duracion_faltante','sum'))
agg.loc[agg['llamadas_sin_duracion'].gt(0),'cant_minutos_llamada'] = np.nan
profile = users.merge(agg,on='user_id',how='left',validate='one_to_one',indicator=True)
profile['sin_actividad_observada'] = profile['_merge'].eq('left_only')
profile = profile.drop(columns='_merge')
profile = profile.merge(plans,left_on='plan',right_on='plan_name',how='left',validate='many_to_one')
assert len(profile)==len(users)
print(profile[['cant_mensajes','cant_llamadas','cant_minutos_llamada']].describe())


## Segmentación explícita
Reglas del ejercicio: bajo si llamadas <5 y mensajes <5; medio si ambas métricas <10 y no cumple bajo; alto en los demás casos con datos completos. Son umbrales descriptivos sobre el periodo observado, no límites contractuales ni segmentos de valor económico. Las edades imputadas se identifican por separado.


In [ ]:
def segmento_uso(llamadas,mensajes):
    if pd.isna(llamadas) or pd.isna(mensajes): return 'Sin actividad observada'
    if llamadas < 5 and mensajes < 5: return 'Bajo'
    if llamadas < 10 and mensajes < 10: return 'Medio'
    return 'Alto'
profile['grupo_uso'] = [segmento_uso(a,b) for a,b in zip(profile['cant_llamadas'],profile['cant_mensajes'])]
profile['grupo_edad'] = np.select(
    [profile['edad_imputada'], profile['age'].lt(30), profile['age'].lt(60)],
    ['Edad imputada','Joven','Adulto'],default='Adulto mayor')
print(profile['grupo_uso'].value_counts(dropna=False))
print(pd.crosstab(profile['plan'],profile['grupo_uso'],normalize='index').round(4))
print(pd.crosstab(profile['grupo_edad'],profile['grupo_uso'],normalize='index').round(4))


## Valores atípicos
IQR señala candidatos a revisión; no confirma error ni legitimidad. No se eliminan automáticamente. La cobertura temporal y la antigüedad de clientes deben revisarse antes de comparar consumos acumulados.


In [ ]:
metrics=['cant_mensajes','cant_llamadas','cant_minutos_llamada']
rows=[]
for col in metrics:
    q1,q3=profile[col].quantile([.25,.75]); iqr=q3-q1
    upper=q3+1.5*iqr
    rows.append({'metrica':col,'q1':q1,'q3':q3,'limite_superior':upper,
                 'usuarios_sobre_limite':int(profile[col].gt(upper).sum())})
print(pd.DataFrame(rows).to_string(index=False))
for col in metrics:
    sns.histplot(data=profile,x=col,hue='plan',stat='density',common_norm=False,
                 element='step',fill=False)
    plt.title(f'Distribución normalizada por plan: {col}')
    plt.tight_layout();plt.show()


## Implicaciones comerciales
Los perfiles son un punto de partida para investigar uso, no oportunidades de upselling confirmadas. Comparar consumo mensual con franquicias, reglas de facturación y precios antes de proponer un cambio. No hay datos de consumo de GB en `usage`; por tanto, no puede estimarse la factura total ni el ahorro completo por plan.

Validar fechas de alta/baja contra eventos y meses de exposición, revisar extremos en la fuente y contrastar necesidades del cliente. No se dispone de resultados de campañas ni se ha demostrado impacto en ingresos o retención.
